Import Libraries

In [1]:
# Cell 1: Imports & paths (minimal)
import os, shutil, random, csv
from typing import Tuple, List
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from transformers import ViTForImageClassification
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, matthews_corrcoef, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# Paths (update if needed)
OUTPUT_ROOT = r"Clients"   # <-- your existing Clients folder
CLIENT_A = os.path.join(OUTPUT_ROOT, "ClientA")
CLIENT_B = os.path.join(OUTPUT_ROOT, "ClientB")
CLIENTS = [CLIENT_A, CLIENT_B]

RESULTS_DIR = "Results"
ROC_DIR = os.path.join(RESULTS_DIR, "roc")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(ROC_DIR, exist_ok=True)

# Training params
IMG_SIZE = 64
BATCH_SIZE = 16
NUM_ROUNDS = 10
LOCAL_EPOCHS = 2
LR = 1e-4
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

required_splits = ["train", "val", "test"]
required_classes = ["monkeypox", "non_monkeypox"]

for c in CLIENTS:
    print(f"\nChecking: {c}")
    if not os.path.isdir(c):
        print("  MISSING client folder!", c)
        continue

    for split in required_splits:
        split_path = os.path.join(c, split)
        if not os.path.isdir(split_path):
            print("  MISSING split folder:", split_path)
            continue

        for cls in required_classes:
            cls_path = os.path.join(split_path, cls)
            if not os.path.isdir(cls_path):
                print("    MISSING class folder:", cls_path)
            else:
                cnt = len([f for f in os.listdir(cls_path)
                           if os.path.isfile(os.path.join(cls_path, f))])
                print(f"    {split}/{cls}: {cnt} files")


C:\Users\KIIT\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.9.1+cpu
Using device: cpu

Checking: Clients\ClientA
    train/monkeypox: 1696 files
    train/non_monkeypox: 2814 files
    val/monkeypox: 566 files
    val/non_monkeypox: 938 files
    test/monkeypox: 566 files
    test/non_monkeypox: 938 files

Checking: Clients\ClientB
    train/monkeypox: 856 files
    train/non_monkeypox: 1058 files
    val/monkeypox: 286 files
    val/non_monkeypox: 353 files
    test/monkeypox: 286 files
    test/non_monkeypox: 353 files


Transforms and Dataset

In [2]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,0.5,0.5),
                         std=(0.5,0.5,0.5)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,0.5,0.5),
                         std=(0.5,0.5,0.5)),
])

class BinaryMonkeypoxDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform

        for cls in os.listdir(root):
            cls_path = os.path.join(root, cls)
            if not os.path.isdir(cls_path):
                continue

            # label mapping: monkeypox -> 1 else 0
            label = 1 if cls.lower().startswith("monkey") else 0

            for f in os.listdir(cls_path):
                fp = os.path.join(cls_path, f)
                if os.path.isfile(fp):
                    self.samples.append((fp, label))

        if len(self.samples) == 0:
            raise RuntimeError(f"No images found inside {root}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


DataLoaders builder & build loaders for all clients

In [3]:
def make_loaders(client_root):
    train_root = os.path.join(client_root, "train")
    val_root   = os.path.join(client_root, "val")
    test_root  = os.path.join(client_root, "test")

    train_ds = BinaryMonkeypoxDataset(train_root, train_transform)
    val_ds   = BinaryMonkeypoxDataset(val_root, val_transform)
    test_ds  = BinaryMonkeypoxDataset(test_root, val_transform)

    # If you get worker issues on Windows, set num_workers=0
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    return train_loader, val_loader, test_loader

# Build loaders
client_train_loaders = []
client_val_loaders = []
client_test_loaders = []

for c in CLIENTS:
    print("\nLoading:", c)
    tr, vl, te = make_loaders(c)
    print(" Train batches:", len(tr))
    print(" Val batches:", len(vl))
    print(" Test batches:", len(te))

    client_train_loaders.append(tr)
    client_val_loaders.append(vl)
    client_test_loaders.append(te)



Loading: Clients\ClientA
 Train batches: 282
 Val batches: 94
 Test batches: 94

Loading: Clients\ClientB
 Train batches: 120
 Val batches: 40
 Test batches: 40


Model helpers & global model

In [4]:
# Cell 4: ViT model helpers and parameter utilities
def get_vit_model(num_labels=2):
    model = ViTForImageClassification.from_pretrained(
        "google/vit-base-patch16-224-in21k",
        num_labels=num_labels,
        image_size=IMG_SIZE,
        ignore_mismatched_sizes=True,
    )
    return model

def get_model_parameters(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

def set_model_parameters(model, params_dict):
    model.load_state_dict(params_dict, strict=True)

def fedavg(params_list):
    avg_params = {}
    with torch.no_grad():
        for key in params_list[0].keys():
            avg_params[key] = torch.stack([p[key] for p in params_list]).mean(0)
    return avg_params

# instantiate global model and loss
global_model = get_vit_model().to(DEVICE)
criterion = nn.CrossEntropyLoss()


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- vit.embeddings.position_embeddings: found shape torch.Size([1, 197, 768]) in the checkpoint and torch.Size([1, 17, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Metrics & ROC utilities (compute_metrics, save_roc_curve, etc.)

In [5]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, matthews_corrcoef, roc_auc_score, roc_curve
)

def compute_metrics(y_true, y_scores, y_pred):
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)
    y_pred = np.array(y_pred)

    if y_true.size == 0:
        return {
            "accuracy": np.nan, "precision": np.nan, "recall": np.nan,
            "f1_score": np.nan, "mcc": np.nan, "auc": np.nan,
            "confusion_matrix": np.array([[0,0],[0,0]])
        }

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    # MCC: require both classes present in y_true and y_pred
    try:
        mcc = float(matthews_corrcoef(y_true, y_pred)) if (len(np.unique(y_true))>1 and len(np.unique(y_pred))>1) else float('nan')
    except Exception:
        mcc = float('nan')

    # AUC: require both classes present in y_true
    try:
        auc = float(roc_auc_score(y_true, y_scores)) if len(np.unique(y_true))>1 else float('nan')
    except Exception:
        auc = float('nan')

    return {
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1_score": float(f1),
        "mcc": mcc,
        "auc": auc,
        "confusion_matrix": cm
    }

def save_roc_curve(y_true, y_scores, path, title="ROC"):
    """
    Save one ROC file. If ROC can't be computed (single class), save a placeholder.
    """
    try:
        if len(np.unique(y_true)) > 1:
            fpr, tpr, _ = roc_curve(y_true, y_scores)
            plt.figure()
            plt.plot(fpr, tpr, linewidth=2)
            plt.plot([0,1],[0,1], linestyle='--')
            plt.xlabel("False Positive Rate")
            plt.ylabel("True Positive Rate")
            plt.title(title)
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(path)
            plt.close()
        else:
            plt.figure(figsize=(4,3))
            plt.text(0.5, 0.5, "ROC not available (single class)", ha='center', va='center')
            plt.axis('off')
            plt.savefig(path)
            plt.close()
    except Exception:
        plt.figure(figsize=(4,3))
        plt.text(0.5, 0.5, "ROC error", ha='center', va='center')
        plt.axis('off')
        plt.savefig(path)
        plt.close()


Training and evaluation functions (train_one_epoch, eval_model)

In [6]:
# Cell 6: training and evaluation helpers
def train_one_epoch(model, loader, optimizer, rnd, client_name, epoch):
    """
    Train for one epoch. DOES NOT save ROC per epoch.
    Returns metrics dict + raw arrays y_true, y_scores, y_pred.
    """
    model.train()
    total_loss = 0.0
    total_samples = 0
    y_trues, y_preds, y_scores = [], [], []

    loop = tqdm(loader, desc=f"Train {client_name}", leave=False)
    for images, labels in loop:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE).long()

        optimizer.zero_grad()
        outputs = model(images)
        logits = outputs.logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        bs = images.size(0)
        total_loss += loss.item() * bs
        total_samples += bs

        probs = torch.softmax(logits, dim=1)[:,1].detach().cpu().numpy()
        preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
        labels_cpu = labels.detach().cpu().numpy()

        y_trues.extend(labels_cpu.tolist())
        y_preds.extend(preds.tolist())
        y_scores.extend(probs.tolist())

    avg_loss = total_loss / total_samples if total_samples > 0 else float('nan')

    metrics = compute_metrics(y_trues, y_scores, y_preds)
    metrics["loss"] = float(avg_loss)

    # return raw arrays for later aggregation and optional single ROC per client/round
    metrics["y_true"] = np.array(y_trues)
    metrics["y_scores"] = np.array(y_scores)
    metrics["y_pred"] = np.array(y_preds)

    return metrics

def eval_model(model, loader, rnd, client_name, mode="val"):
    """
    Evaluate and SAVE ONE ROC per client per round.
    Returns metrics dict + raw arrays.
    """
    model.eval()
    total_loss = 0.0
    total_samples = 0
    y_trues, y_preds, y_scores = [], [], []

    with torch.no_grad():
        loop = tqdm(loader, desc=f"{mode.capitalize()} {client_name}", leave=False)
        for images, labels in loop:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE).long()

            outputs = model(images)
            logits = outputs.logits
            loss = criterion(logits, labels)

            bs = images.size(0)
            total_loss += loss.item() * bs
            total_samples += bs

            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            labels_cpu = labels.cpu().numpy()

            y_trues.extend(labels_cpu.tolist())
            y_preds.extend(preds.tolist())
            y_scores.extend(probs.tolist())

    avg_loss = total_loss / total_samples if total_samples > 0 else float('nan')

    metrics = compute_metrics(y_trues, y_scores, y_preds)
    metrics["loss"] = float(avg_loss)

    # Save one ROC per client per round for validation/test
    fname = f"ROC_round{rnd+1}_{client_name}_{mode}.png" if rnd is not None else f"{client_name}_{mode}.png"
    roc_path = os.path.join(ROC_DIR, fname)
    save_roc_curve(y_trues, y_scores, roc_path, title=f"{client_name} {mode.upper()} R{rnd+1 if rnd is not None else ''}")
    metrics["roc_path"] = roc_path

    metrics["y_true"] = np.array(y_trues)
    metrics["y_scores"] = np.array(y_scores)
    metrics["y_pred"] = np.array(y_preds)

    return metrics


Federated training loop, aggregation, CSV save, final test eval

In [7]:
SAVE_MODEL_DIR = RESULTS_DIR
os.makedirs(SAVE_MODEL_DIR, exist_ok=True)

# containers for per-round storage (means across clients + concatenated arrays)
training_metrics_all = {
    "loss": [], "accuracy": [], "precision": [], "recall": [], "f1": [], "mcc": [], "auc": [], "confusion_matrix": [],
    "y_true_concat": [], "y_scores_concat": [], "y_pred_concat": []
}
validation_metrics_all = {
    "loss": [], "accuracy": [], "precision": [], "recall": [], "f1": [], "mcc": [], "auc": [], "confusion_matrix": [],
    "y_true_concat": [], "y_scores_concat": [], "y_pred_concat": []
}

for rnd in range(NUM_ROUNDS):
    print(f"\n========== Federated Round {rnd+1}/{NUM_ROUNDS} ==========")

    global_params = get_model_parameters(global_model)
    client_params_list = []

    # will collect each client's concatenated train arrays (across epochs) for agg
    client_train_concat_trues = []
    client_train_concat_scores = []
    client_train_concat_preds = []

    # last-epoch metrics for each client (to compute per-client means)
    round_local_train_metrics = []

    # -------- Local training on each client --------
    for idx, (train_loader, val_loader) in enumerate(zip(client_train_loaders, client_val_loaders)):
        client_name = f"Client{idx+1}"
        print(f"\n--> {client_name} local training")

        local_model = get_vit_model(num_labels=2).to(DEVICE)
        set_model_parameters(local_model, global_params)
        optimizer = AdamW(local_model.parameters(), lr=LR)

        # keep per-epoch lists so we can aggregate across epochs for this client
        epoch_trues = []
        epoch_scores = []
        epoch_preds = []
        last_metrics = None

        for ep in range(LOCAL_EPOCHS):
            metrics = train_one_epoch(local_model, train_loader, optimizer, rnd, client_name, ep)
            # print per-epoch nicely (label "Epochs X")
            print(f"   Epochs {ep+1}: Loss={metrics['loss']:.4f}, Acc={metrics['accuracy']:.4f}, "
                  f"Prec={metrics['precision']:.4f}, Rec={metrics['recall']:.4f}, F1={metrics['f1_score']:.4f}, "
                  f"MCC={np.nan if np.isnan(metrics['mcc']) else metrics['mcc']:.4f}, "
                  f"AUC={np.nan if np.isnan(metrics['auc']) else metrics['auc']:.4f}")

            # accumulate arrays for this epoch
            epoch_trues.append(metrics["y_true"])
            epoch_scores.append(metrics["y_scores"])
            epoch_preds.append(metrics["y_pred"])

            last_metrics = metrics

        # concatenate this client's epoch arrays -> one train-set for this client this round
        client_true_concat = np.concatenate(epoch_trues) if len(epoch_trues) > 0 else np.array([])
        client_scores_concat = np.concatenate(epoch_scores) if len(epoch_scores) > 0 else np.array([])
        client_preds_concat = np.concatenate(epoch_preds) if len(epoch_preds) > 0 else np.array([])

        # save a single ROC for the client's training data this round (NOT per epoch)
        if client_true_concat.size > 0:
            train_roc_path = os.path.join(ROC_DIR, f"ROC_round{rnd+1}_{client_name}_train.png")
            save_roc_curve(client_true_concat, client_scores_concat, train_roc_path, title=f"{client_name} TRAIN R{rnd+1}")
        else:
            train_roc_path = None

        # push to lists for round-level aggregation
        client_train_concat_trues.append(client_true_concat)
        client_train_concat_scores.append(client_scores_concat)
        client_train_concat_preds.append(client_preds_concat)

        # last epoch metrics used in per-client mean
        round_local_train_metrics.append(last_metrics)
        client_params_list.append(get_model_parameters(local_model))

    # -------- Federated averaging ----------
    print("\n--> Aggregating client models with FedAvg")
    new_global_params = fedavg(client_params_list)
    set_model_parameters(global_model, new_global_params)

    # -------- Validation (evaluate global on each client's val set) ----------
    round_val_metrics = []
    client_val_concat_trues = []
    client_val_concat_scores = []
    client_val_concat_preds = []

    for idx, val_loader in enumerate(client_val_loaders):
        client_name = f"Client{idx+1}"
        vmetrics = eval_model(global_model, val_loader, rnd, client_name, mode="val")
        print(f"   {client_name} VAL: Loss={vmetrics['loss']:.4f}, Acc={vmetrics['accuracy']:.4f}, "
              f"Prec={vmetrics['precision']:.4f}, Rec={vmetrics['recall']:.4f}, F1={vmetrics['f1_score']:.4f}, "
              f"MCC={np.nan if np.isnan(vmetrics['mcc']) else vmetrics['mcc']:.4f}, "
              f"AUC={np.nan if np.isnan(vmetrics['auc']) else vmetrics['auc']:.4f}")
        round_val_metrics.append(vmetrics)

        # store val arrays for aggregation
        client_val_concat_trues.append(vmetrics["y_true"])
        client_val_concat_scores.append(vmetrics["y_scores"])
        client_val_concat_preds.append(vmetrics["y_pred"])

    # -------- PER-CLIENT MEANS (store) ----------
    training_metrics_all["loss"].append(np.mean([m["loss"] for m in round_local_train_metrics]))
    training_metrics_all["accuracy"].append(np.mean([m["accuracy"] for m in round_local_train_metrics]))
    training_metrics_all["precision"].append(np.mean([m["precision"] for m in round_local_train_metrics]))
    training_metrics_all["recall"].append(np.mean([m["recall"] for m in round_local_train_metrics]))
    training_metrics_all["f1"].append(np.mean([m["f1_score"] for m in round_local_train_metrics]))
    training_metrics_all["mcc"].append(np.nanmean([m["mcc"] for m in round_local_train_metrics]))
    training_metrics_all["auc"].append(np.nanmean([m["auc"] for m in round_local_train_metrics]))
    training_metrics_all["confusion_matrix"].append(sum([m["confusion_matrix"] for m in round_local_train_metrics]))

    validation_metrics_all["loss"].append(np.mean([m["loss"] for m in round_val_metrics]))
    validation_metrics_all["accuracy"].append(np.mean([m["accuracy"] for m in round_val_metrics]))
    validation_metrics_all["precision"].append(np.mean([m["precision"] for m in round_val_metrics]))
    validation_metrics_all["recall"].append(np.mean([m["recall"] for m in round_val_metrics]))
    validation_metrics_all["f1"].append(np.mean([m["f1_score"] for m in round_val_metrics]))
    validation_metrics_all["mcc"].append(np.nanmean([m["mcc"] for m in round_val_metrics]))
    validation_metrics_all["auc"].append(np.nanmean([m["auc"] for m in round_val_metrics]))
    validation_metrics_all["confusion_matrix"].append(sum([m["confusion_matrix"] for m in round_val_metrics]))

    # -------- AGGREGATED (concatenate across clients) TRAIN metrics for the round ----------
    try:
        all_y_true_train = np.concatenate([a for a in client_train_concat_trues if a.size>0]) if len(client_train_concat_trues)>0 else np.array([])
        all_y_scores_train = np.concatenate([a for a in client_train_concat_scores if a.size>0]) if len(client_train_concat_scores)>0 else np.array([])
        all_y_pred_train = np.concatenate([a for a in client_train_concat_preds if a.size>0]) if len(client_train_concat_preds)>0 else np.array([])
    except Exception as e:
        print("Warning concatenating train arrays:", e)
        all_y_true_train = np.array([]); all_y_scores_train = np.array([]); all_y_pred_train = np.array([])

    if all_y_true_train.size > 0:
        agg_train = compute_metrics(all_y_true_train, all_y_scores_train, all_y_pred_train)
        # include loss for aggregated train: compute combined loss as not straightforward per-sample; here we'll print per-client-mean loss and also store NaN for agg_loss.
        print(f"--> ROUND {rnd+1} AGG TRAIN: Loss(mean)={np.mean([m['loss'] for m in round_local_train_metrics]):.4f}, "
              f"Acc={agg_train['accuracy']:.4f}, Prec={agg_train['precision']:.4f}, Rec={agg_train['recall']:.4f}, "
              f"F1={agg_train['f1_score']:.4f}, MCC={np.nan if np.isnan(agg_train['mcc']) else agg_train['mcc']:.4f}, "
              f"AUC={np.nan if np.isnan(agg_train['auc']) else agg_train['auc']:.4f}")
        # store concatenated arrays for CSV aggregate computation later
        training_metrics_all["y_true_concat"].append(all_y_true_train)
        training_metrics_all["y_scores_concat"].append(all_y_scores_train)
        training_metrics_all["y_pred_concat"].append(all_y_pred_train)
    else:
        print(f"--> ROUND {rnd+1} AGG TRAIN: No concatenated train data available (cannot compute agg metrics).")

    # -------- AGGREGATED (concatenate across clients) VAL metrics for the round ----------
    try:
        all_y_true_val = np.concatenate([a for a in client_val_concat_trues if a.size>0]) if len(client_val_concat_trues)>0 else np.array([])
        all_y_scores_val = np.concatenate([a for a in client_val_concat_scores if a.size>0]) if len(client_val_concat_scores)>0 else np.array([])
        all_y_pred_val = np.concatenate([a for a in client_val_concat_preds if a.size>0]) if len(client_val_concat_preds)>0 else np.array([])
    except Exception as e:
        print("Warning concatenating val arrays:", e)
        all_y_true_val = np.array([]); all_y_scores_val = np.array([]); all_y_pred_val = np.array([])

    if all_y_true_val.size > 0:
        agg_val = compute_metrics(all_y_true_val, all_y_scores_val, all_y_pred_val)
        print(f"--> ROUND {rnd+1} AGG VAL: Loss(mean)={np.mean([m['loss'] for m in round_val_metrics]):.4f}, "
              f"Acc={agg_val['accuracy']:.4f}, Prec={agg_val['precision']:.4f}, Rec={agg_val['recall']:.4f}, "
              f"F1={agg_val['f1_score']:.4f}, MCC={np.nan if np.isnan(agg_val['mcc']) else agg_val['mcc']:.4f}, "
              f"AUC={np.nan if np.isnan(agg_val['auc']) else agg_val['auc']:.4f}")
        validation_metrics_all["y_true_concat"].append(all_y_true_val)
        validation_metrics_all["y_scores_concat"].append(all_y_scores_val)
        validation_metrics_all["y_pred_concat"].append(all_y_pred_val)
    else:
        print(f"--> ROUND {rnd+1} AGG VAL: No concatenated val data available (cannot compute agg metrics).")

# ---------- End of federated rounds ----------

# ----- Save final global model to Drive -----
model_save_path = os.path.join(SAVE_MODEL_DIR, "vit_monkeypox_federated.pth")
torch.save(global_model.state_dict(), model_save_path)
print("\nSaved federated global model to:", model_save_path)

# ----- Final test evaluation (per-client) -----
print("\n=== FINAL TEST EVALUATION ===")
final_test_results = []
for idx, test_loader in enumerate(client_test_loaders):
    cname = f"Client{idx+1}"
    tmetrics = eval_model(global_model, test_loader, rnd, cname, mode="test")
    print(f"{cname} TEST: Loss={tmetrics['loss']:.4f}, Acc={tmetrics['accuracy']:.4f}, "
          f"Prec={tmetrics['precision']:.4f}, Rec={tmetrics['recall']:.4f}, F1={tmetrics['f1_score']:.4f}, "
          f"MCC={np.nan if np.isnan(tmetrics['mcc']) else tmetrics['mcc']:.4f}, "
          f"AUC={np.nan if np.isnan(tmetrics['auc']) else tmetrics['auc']:.4f}")
    final_test_results.append((cname, tmetrics))

# ----- Save CSV summary to Drive -----
csv_path = os.path.join(SAVE_MODEL_DIR, "summary_metrics_rounds.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    # header
    writer.writerow([
        "round",
        # per-client-mean train
        "train_loss_mean","train_acc_mean","train_prec_mean","train_rec_mean","train_f1_mean","train_mcc_mean","train_auc_mean",
        # agg train (concatenated)
        "train_agg_acc","train_agg_prec","train_agg_rec","train_agg_f1","train_agg_mcc","train_agg_auc",
        # per-client-mean val
        "val_loss_mean","val_acc_mean","val_prec_mean","val_rec_mean","val_f1_mean","val_mcc_mean","val_auc_mean",
        # agg val (concatenated)
        "val_agg_acc","val_agg_prec","val_agg_rec","val_agg_f1","val_agg_mcc","val_agg_auc"
    ])

    for r in range(NUM_ROUNDS):
        # aggregated train metrics (from concatenated arrays) if available
        if r < len(training_metrics_all["y_true_concat"]):
            agg_tr = compute_metrics(
                training_metrics_all["y_true_concat"][r],
                training_metrics_all["y_scores_concat"][r],
                training_metrics_all["y_pred_concat"][r]
            )
        else:
            agg_tr = {"accuracy": np.nan, "precision": np.nan, "recall": np.nan, "f1_score": np.nan, "mcc": np.nan, "auc": np.nan}

        # aggregated val metrics
        if r < len(validation_metrics_all["y_true_concat"]):
            agg_v = compute_metrics(
                validation_metrics_all["y_true_concat"][r],
                validation_metrics_all["y_scores_concat"][r],
                validation_metrics_all["y_pred_concat"][r]
            )
        else:
            agg_v = {"accuracy": np.nan, "precision": np.nan, "recall": np.nan, "f1_score": np.nan, "mcc": np.nan, "auc": np.nan}

        writer.writerow([
            r+1,
            training_metrics_all["loss"][r], training_metrics_all["accuracy"][r], training_metrics_all["precision"][r],
            training_metrics_all["recall"][r], training_metrics_all["f1"][r], training_metrics_all["mcc"][r], training_metrics_all["auc"][r],
            agg_tr["accuracy"], agg_tr["precision"], agg_tr["recall"], agg_tr["f1_score"], agg_tr["mcc"], agg_tr["auc"],
            validation_metrics_all["loss"][r], validation_metrics_all["accuracy"][r], validation_metrics_all["precision"][r],
            validation_metrics_all["recall"][r], validation_metrics_all["f1"][r], validation_metrics_all["mcc"][r], validation_metrics_all["auc"][r],
            agg_v["accuracy"], agg_v["precision"], agg_v["recall"], agg_v["f1_score"], agg_v["mcc"], agg_v["auc"]
        ])

    # append final test summary
    writer.writerow([])
    writer.writerow(["final_test"])
    writer.writerow(["client","loss","acc","precision","recall","f1","mcc","auc"])
    for cname, tm in final_test_results:
        writer.writerow([cname, tm["loss"], tm["accuracy"], tm["precision"], tm["recall"], tm["f1_score"], tm["mcc"], tm["auc"]])

print("\nCSV saved at:", csv_path)



========== Federated Round 1/10 ==========

--> Client1 local training


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- vit.embeddings.position_embeddings: found shape torch.Size([1, 197, 768]) in the checkpoint and torch.Size([1, 17, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   Epochs 1: Loss=0.4148, Acc=0.8082, Prec=0.7672, Rec=0.7034, F1=0.7339, MCC=0.5858, AUC=0.8825


   Epochs 2: Loss=0.2044, Acc=0.9228, Prec=0.8974, Rec=0.8974, F1=0.8974, MCC=0.8356, AUC=0.9721

--> Client2 local training


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- vit.embeddings.position_embeddings: found shape torch.Size([1, 197, 768]) in the checkpoint and torch.Size([1, 17, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

   Epochs 2: Loss=0.0295, Acc=0.9896, Prec=0.9883, Rec=0.9883, F1=0.9883, MCC=0.9789, AUC=0.9993

--> Aggregating client models with FedAvg


   Client1 VAL: Loss=0.0397, Acc=0.9860, Prec=0.9723, Rec=0.9912, F1=0.9816, MCC=0.9705, AUC=0.9980


   Client2 VAL: Loss=0.0522, Acc=0.9781, Prec=0.9564, Rec=0.9965, F1=0.9760, MCC=0.9566, AUC=0.9996
--> ROUND 10 AGG TRAIN: Loss(mean)=0.0295, Acc=0.9888, Prec=0.9859, Rec=0.9859, F1=0.9859, MCC=0.9766, AUC=0.9987
--> ROUND 10 AGG VAL: Loss(mean)=0.0460, Acc=0.9837, Prec=0.9669, Rec=0.9930, F1=0.9797, MCC=0.9663, AUC=0.9985

Saved federated global model to: Results\vit_monkeypox_federated.pth

=== FINAL TEST EVALUATION ===


Client1 TEST: Loss=0.0380, Acc=0.9874, Prec=0.9807, Rec=0.9859, F1=0.9833, MCC=0.9731, AUC=0.9979


Client2 TEST: Loss=0.0603, Acc=0.9765, Prec=0.9502, Rec=1.0000, F1=0.9744, MCC=0.9538, AUC=0.9999

CSV saved at: Results\summary_metrics_rounds.csv


Image Testing

In [9]:
BINARY_CLASS_NAMES = ["non_monkeypox", "monkeypox"]

IMG_SIZE = 64

infer_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

def get_vit_model(num_labels: int = 2):
    return ViTForImageClassification.from_pretrained(
        "google/vit-base-patch16-224-in21k",
        num_labels=num_labels,
        image_size=IMG_SIZE,
        ignore_mismatched_sizes=True,
    )

model_path = "Results/vit_monkeypox_federated.pth"

fed_model = get_vit_model(num_labels=2).to(DEVICE)
fed_model.load_state_dict(torch.load(model_path, map_location=DEVICE))
fed_model.eval()
print("Federated ViT model loaded successfully from:", model_path)

def predict_image(model, image_path: str):
    img = Image.open(image_path).convert("RGB")
    x = infer_transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(x).logits
        probs = torch.softmax(logits, dim=1)[0]

    pred_idx = probs.argmax().item()
    pred_class = BINARY_CLASS_NAMES[pred_idx]
    confidence = float(probs[pred_idx])
    return pred_class, confidence

test_image_path = "Clients/ClientA/test/monkeypox/MKP_01_04_10.jpg"

if os.path.exists(test_image_path):
    pcls, conf = predict_image(fed_model, test_image_path)
    print("Prediction:", pcls)
    print("Confidence:", conf)
else:
    print("Update 'test_image_path' to point to a valid image on Drive.")

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- vit.embeddings.position_embeddings: found shape torch.Size([1, 197, 768]) in the checkpoint and torch.Size([1, 17, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Federated ViT model loaded successfully from: Results/vit_monkeypox_federated.pth
Prediction: monkeypox
Confidence: 0.9987472295761108


In [10]:
fed_model = get_vit_model(num_labels=2).to(DEVICE)
fed_model.load_state_dict(torch.load(model_path, map_location=DEVICE))
fed_model.eval()
print("Federated ViT model loaded successfully from:", model_path)

infer_transform = val_transform

def predict_image(model, image_path: str):
    """
    Predict monkeypox / non_monkeypox for a single image.
    Returns: (predicted_class_name, confidence_score)
    """
    img = Image.open(image_path).convert("RGB")
    x = infer_transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(x)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)[0]

    pred_idx = probs.argmax().item()
    pred_class = BINARY_CLASS_NAMES[pred_idx]
    confidence = float(probs[pred_idx])
    return pred_class, confidence

test_image_path = "Clients/ClientA/test/non_monkeypox/CHP_04_01_10.jpg"

if os.path.exists(test_image_path):
    pcls, conf = predict_image(fed_model, test_image_path)
    print("Prediction:", pcls)
    print("Confidence:", conf)
else:
    print("Update 'test_image_path' to point to a valid image on Drive.")

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- vit.embeddings.position_embeddings: found shape torch.Size([1, 197, 768]) in the checkpoint and torch.Size([1, 17, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Federated ViT model loaded successfully from: Results/vit_monkeypox_federated.pth
Prediction: non_monkeypox
Confidence: 0.9987861514091492


Setup, imports, load CSV summary (if available)

In [15]:
# Cell 1: setup & CSV fallback (reads MCC now)
import os, sys, csv
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

RESULTS_DIR = "Results"
MODEL_PATH = os.path.join(RESULTS_DIR, "vit_monkeypox_federated.pth")
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots_from_model")
os.makedirs(PLOTS_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Model path exists:", os.path.exists(MODEL_PATH))

# CSV summary parsing (reads train/val: loss, acc, prec, rec, f1, mcc, auc)
csv_path = os.path.join(RESULTS_DIR, "summary_metrics_rounds.csv")
csv_summary = None
if os.path.exists(csv_path):
    csv_summary = {"round": [],
                   "train_loss_mean": [], "train_acc_mean": [], "train_prec_mean": [], "train_rec_mean": [], "train_f1_mean": [], "train_mcc_mean": [], "train_auc_mean": [],
                   "val_loss_mean": [],   "val_acc_mean": [],   "val_prec_mean": [],   "val_rec_mean": [],   "val_f1_mean": [],   "val_mcc_mean": [],   "val_auc_mean": []}
    with open(csv_path, "r") as f:
        rows = list(csv.reader(f))
    hdr_idx = None
    for i, r in enumerate(rows):
        if len(r) > 0 and r[0].strip().lower() == "round":
            hdr_idx = i
            break
    if hdr_idx is not None:
        for r in rows[hdr_idx+1:]:
            if len(r) == 0 or r[0].strip() == "":
                break
            try:
                csv_summary["round"].append(int(r[0]))
                # train: columns as written in your writer
                csv_summary["train_loss_mean"].append(float(r[1]))
                csv_summary["train_acc_mean"].append(float(r[2]))
                csv_summary["train_prec_mean"].append(float(r[3]))
                csv_summary["train_rec_mean"].append(float(r[4]))
                csv_summary["train_f1_mean"].append(float(r[5]))
                csv_summary["train_mcc_mean"].append(float(r[6]))
                csv_summary["train_auc_mean"].append(float(r[7]))
                # val: columns start at index 11 in your writer
                csv_summary["val_loss_mean"].append(float(r[11]))
                csv_summary["val_acc_mean"].append(float(r[12]))
                csv_summary["val_prec_mean"].append(float(r[13]))
                csv_summary["val_rec_mean"].append(float(r[14]))
                csv_summary["val_f1_mean"].append(float(r[15]))
                csv_summary["val_mcc_mean"].append(float(r[16]))
                csv_summary["val_auc_mean"].append(float(r[17]))
            except Exception:
                # if any parsing error, fill with nan
                csv_summary["train_loss_mean"].append(np.nan)
                csv_summary["train_acc_mean"].append(np.nan)
                csv_summary["train_prec_mean"].append(np.nan)
                csv_summary["train_rec_mean"].append(np.nan)
                csv_summary["train_f1_mean"].append(np.nan)
                csv_summary["train_mcc_mean"].append(np.nan)
                csv_summary["train_auc_mean"].append(np.nan)
                csv_summary["val_loss_mean"].append(np.nan)
                csv_summary["val_acc_mean"].append(np.nan)
                csv_summary["val_prec_mean"].append(np.nan)
                csv_summary["val_rec_mean"].append(np.nan)
                csv_summary["val_f1_mean"].append(np.nan)
                csv_summary["val_mcc_mean"].append(np.nan)
                csv_summary["val_auc_mean"].append(np.nan)
    print("Loaded CSV summary rounds:", len(csv_summary["round"]))
else:
    print("No CSV summary found at", csv_path)


Device: cpu
Model path exists: True
Loaded CSV summary rounds: 10


Load model (ViT) from saved .pth (safe load)

In [12]:
# Cell 2: load model skeleton and weights (ViT)
from transformers import ViTForImageClassification

# Create ViT model instance (match what you used during training)
IMG_SIZE = 64   # set to whatever you trained with; your original Cell 1 used 64
try:
    model = ViTForImageClassification.from_pretrained(
        "google/vit-base-patch16-224-in21k",
        num_labels=2,
        image_size=IMG_SIZE,
        ignore_mismatched_sizes=True
    )
    # load saved state dict (may have keys matching ViT)
    state = torch.load(MODEL_PATH, map_location=DEVICE)
    # if saved as state_dict directly, load it; if saved as model.state_dict(), works too
    try:
        model.load_state_dict(state, strict=False)
        print("Loaded state_dict into ViT (strict=False).")
    except Exception as e:
        # sometimes saved file contains only model dict or wrapper
        if isinstance(state, dict) and all(isinstance(v, torch.Tensor) for v in state.values()):
            model.load_state_dict(state, strict=False)
            print("Loaded state dict (fallback).")
        else:
            print("Warning: couldn't directly load state dict into ViT:", e)
except Exception as e:
    print("Could not instantiate/load ViT model:", e)
    model = None

if model is not None:
    model.to(DEVICE)
    model.eval()


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- vit.embeddings.position_embeddings: found shape torch.Size([1, 197, 768]) in the checkpoint and torch.Size([1, 17, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded state_dict into ViT (strict=False).


In [16]:
from transformers import ViTForImageClassification
import torch
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "Results/vit_monkeypox_federated.pth"

IMG_SIZE = 64  # MUST MATCH what you trained with

print("Loading model from:", MODEL_PATH)

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=2,
    image_size=IMG_SIZE,
    ignore_mismatched_sizes=True,
)

state = torch.load(MODEL_PATH, map_location=DEVICE)

try:
    model.load_state_dict(state, strict=False)
    print("Model weights loaded successfully.")
except Exception as e:
    print("⚠️ Warning: could not load state_dict strictly:", e)
    print("Trying fallback loading…")
    model.load_state_dict(state, strict=False)

model.to(DEVICE)
model.eval()

print("Model loaded and set to eval mode on", DEVICE)


Loading model from: Results/vit_monkeypox_federated.pth


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- vit.embeddings.position_embeddings: found shape torch.Size([1, 197, 768]) in the checkpoint and torch.Size([1, 17, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model weights loaded successfully.
Model loaded and set to eval mode on cpu


Build per-round lists (include MCC)

In [17]:
# Cell 2: build per-round lists for train & val (now including MCC)
# Output lists will be used by the plotting cell.

# Initialize (defaults)
round_losses_train, round_losses_val = [], []
round_accuracies_train, round_accuracies_val = [], []
round_precisions_train, round_precisions_val = [], []
round_recalls_train, round_recalls_val = [], []
round_f1s_train, round_f1s_val = [], []
round_aucs_train, round_aucs_val = [], []
round_mccs_train, round_mccs_val = [], []

# Try in-memory metrics first (training_metrics_all / validation_metrics_all)
if ('training_metrics_all' in globals()) and ('validation_metrics_all' in globals()) and training_metrics_all is not None and validation_metrics_all is not None:
    round_losses_train = list(training_metrics_all.get("loss", []))
    round_accuracies_train = list(training_metrics_all.get("accuracy", []))
    round_precisions_train = list(training_metrics_all.get("precision", []))
    round_recalls_train = list(training_metrics_all.get("recall", []))
    round_f1s_train = list(training_metrics_all.get("f1", []))
    round_aucs_train = list(training_metrics_all.get("auc", []))
    round_mccs_train = list(training_metrics_all.get("mcc", []))

    round_losses_val = list(validation_metrics_all.get("loss", []))
    round_accuracies_val = list(validation_metrics_all.get("accuracy", []))
    round_precisions_val = list(validation_metrics_all.get("precision", []))
    round_recalls_val = list(validation_metrics_all.get("recall", []))
    round_f1s_val = list(validation_metrics_all.get("f1", []))
    round_aucs_val = list(validation_metrics_all.get("auc", []))
    round_mccs_val = list(validation_metrics_all.get("mcc", []))

    print("Using in-memory training/validation metrics (found).")
# Else fallback to CSV summary (if present)
elif csv_summary is not None and len(csv_summary["round"]) > 0:
    round_losses_train = csv_summary["train_loss_mean"]
    round_accuracies_train = csv_summary["train_acc_mean"]
    round_precisions_train = csv_summary["train_prec_mean"]
    round_recalls_train = csv_summary["train_rec_mean"]
    round_f1s_train = csv_summary["train_f1_mean"]
    round_aucs_train = csv_summary["train_auc_mean"]
    round_mccs_train = csv_summary["train_mcc_mean"]

    round_losses_val = csv_summary["val_loss_mean"]
    round_accuracies_val = csv_summary["val_acc_mean"]
    round_precisions_val = csv_summary["val_prec_mean"]
    round_recalls_val = csv_summary["val_rec_mean"]
    round_f1s_val = csv_summary["val_f1_mean"]
    round_aucs_val = csv_summary["val_auc_mean"]
    round_mccs_val = csv_summary["val_mcc_mean"]

    print("Using CSV summary metrics (fallback).")
else:
    print("No per-round training/validation metrics found in memory or CSV. All lists are empty and plotting will be skipped.")

# Quick sanity print
print("Rounds (train):", len(round_losses_train), "Rounds (val):", len(round_losses_val))


Using in-memory training/validation metrics (found).
Rounds (train): 10 Rounds (val): 10


Plot each metric separately for train & val (includes MCC)

In [18]:
# Cell 3: plot separate graphs for each metric including MCC

import matplotlib.pyplot as plt
from matplotlib import ticker
import os

def plot_and_save(values, ylabel, filename):
    if values is None or len(values) == 0:
        print(f"No data for {ylabel}; skipping.")
        return
    rounds = list(range(1, len(values) + 1))
    save_path = os.path.join(PLOTS_DIR, filename)
    plt.figure(figsize=(6, 4))
    plt.plot(rounds, values, marker='o', linewidth=2)
    plt.title(f"{ylabel} Across Federated Rounds")
    plt.xlabel("Federated Round")
    plt.ylabel(ylabel)
    plt.grid(True)
    plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved {ylabel} -> {save_path}")

# Loss
plot_and_save(round_losses_train, "Loss (Train)", "train_loss.png")
plot_and_save(round_losses_val,   "Loss (Val)",   "val_loss.png")

# Accuracy
plot_and_save(round_accuracies_train, "Accuracy (Train)", "train_accuracy.png")
plot_and_save(round_accuracies_val,   "Accuracy (Val)",   "val_accuracy.png")

# Precision
plot_and_save(round_precisions_train, "Precision (Train)", "train_precision.png")
plot_and_save(round_precisions_val,   "Precision (Val)",   "val_precision.png")

# Recall
plot_and_save(round_recalls_train, "Recall (Train)", "train_recall.png")
plot_and_save(round_recalls_val,   "Recall (Val)",   "val_recall.png")

# F1
plot_and_save(round_f1s_train, "F1 Score (Train)", "train_f1.png")
plot_and_save(round_f1s_val,   "F1 Score (Val)",   "val_f1.png")

# AUC
plot_and_save(round_aucs_train, "AUC (Train)", "train_auc.png")
plot_and_save(round_aucs_val,   "AUC (Val)",   "val_auc.png")

# MCC (new)
plot_and_save(round_mccs_train, "MCC (Train)", "train_mcc.png")
plot_and_save(round_mccs_val,   "MCC (Val)",   "val_mcc.png")

print("All requested metric plots saved to:", PLOTS_DIR)


Saved Loss (Train) -> Results\plots_from_model\train_loss.png
Saved Loss (Val) -> Results\plots_from_model\val_loss.png
Saved Accuracy (Train) -> Results\plots_from_model\train_accuracy.png
Saved Accuracy (Val) -> Results\plots_from_model\val_accuracy.png
Saved Precision (Train) -> Results\plots_from_model\train_precision.png
Saved Precision (Val) -> Results\plots_from_model\val_precision.png
Saved Recall (Train) -> Results\plots_from_model\train_recall.png
Saved Recall (Val) -> Results\plots_from_model\val_recall.png
Saved F1 Score (Train) -> Results\plots_from_model\train_f1.png
Saved F1 Score (Val) -> Results\plots_from_model\val_f1.png
Saved AUC (Train) -> Results\plots_from_model\train_auc.png
Saved AUC (Val) -> Results\plots_from_model\val_auc.png
Saved MCC (Train) -> Results\plots_from_model\train_mcc.png
Saved MCC (Val) -> Results\plots_from_model\val_mcc.png
All requested metric plots saved to: Results\plots_from_model


COnfusion Matrix

In [19]:
RESULTS_DIR = "Results"
MODEL_PATH = os.path.join(RESULTS_DIR, "vit_monkeypox_federated.pth")

# SAVE HERE (same folder used earlier for metric graphs)
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots_from_model")
os.makedirs(PLOTS_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BINARY_CLASS_NAMES = ("non_monkeypox", "monkeypox")

# ------------------ Plot Confusion Matrix ------------------
def plot_confusion_matrix(cm, classes, title, save_path):
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.title(title)
    plt.colorbar(fraction=0.046, pad=0.04)

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    # Always black numbers
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j, i, str(int(cm[i, j])),
                ha="center", color="black", fontsize=12, fontweight="bold"
            )

    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved Confusion Matrix:", save_path)

# ------------------ Prediction-only evaluation ------------------
def eval_confusion_only(model, loader, device=DEVICE):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Evaluating", leave=False):
            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)
            logits = outputs.logits if hasattr(outputs, "logits") else outputs
            preds = torch.argmax(logits, dim=1)

            y_true.extend(labels.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())

    return confusion_matrix(np.array(y_true), np.array(y_pred))

# ------------------ Load Model ------------------
if "fed_model" in globals():
    model = fed_model
    model.to(DEVICE)
    model.eval()
    print("Using existing fed_model")
else:
    print("Loading ViT model from:", MODEL_PATH)
    IMG_SIZE = 64  # match your training size
    model = ViTForImageClassification.from_pretrained(
        "google/vit-base-patch16-224-in21k",
        num_labels=2,
        image_size=IMG_SIZE,
        ignore_mismatched_sizes=True,
    )
    state = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(state, strict=False)
    model.to(DEVICE)
    model.eval()
    print("Model loaded")

# ------------------ Evaluate each client and save CM ------------------
if "client_val_loaders" not in globals():
    raise RuntimeError("client_val_loaders not found. Please run your data loader cell first.")

for idx, val_loader in enumerate(client_val_loaders):
    client_name = f"Client{idx+1}"
    print(f"\n=== Creating Confusion Matrix for {client_name} ===")

    cm = eval_confusion_only(model, val_loader)

    # Print exactly like your example:
    print("Confusion Matrix:")
    print(cm)

    save_path = os.path.join(PLOTS_DIR, f"fed_vit_confusion_matrix_{client_name}.png")

    plot_confusion_matrix(
        cm,
        BINARY_CLASS_NAMES,
        title=f"Confusion Matrix - {client_name}",
        save_path=save_path
    )

print("\nAll confusion matrices saved to:", PLOTS_DIR)

Using existing fed_model

=== Creating Confusion Matrix for Client1 ===


Confusion Matrix:
[[922  16]
 [  5 561]]
Saved Confusion Matrix: Results\plots_from_model\fed_vit_confusion_matrix_Client1.png

=== Creating Confusion Matrix for Client2 ===


Confusion Matrix:
[[340  13]
 [  1 285]]
Saved Confusion Matrix: Results\plots_from_model\fed_vit_confusion_matrix_Client2.png

All confusion matrices saved to: Results\plots_from_model


ROC Curve/Graph

In [20]:
RESULTS_DIR = "Results"
MODEL_PATH = os.path.join(RESULTS_DIR, "vit_monkeypox_federated.pth")
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots_from_model")
os.makedirs(PLOTS_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ------------- helper: evaluate loader to get y_true and y_scores -------------
def eval_get_scores(model, loader, device=DEVICE):
    model.eval()
    y_true = []
    y_scores = []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Scoring", leave=False):
            imgs = imgs.to(device)
            labels = labels.to(device).long()
            outputs = model(imgs)
            logits = outputs.logits if hasattr(outputs, "logits") else outputs
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()  # positive class prob
            y_true.extend(labels.cpu().numpy().tolist())
            y_scores.extend(probs.tolist())
    return np.array(y_true), np.array(y_scores)

# ------------- helper: plot ROC and save (placeholder if single-class) -------------
def plot_roc_and_save(y_true, y_scores, out_path, title=None):
    if y_true.size == 0:
        # nothing — create placeholder
        plt.figure(figsize=(4,3))
        plt.text(0.5, 0.5, "ROC not available (no samples)", ha='center', va='center')
        plt.axis('off')
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close()
        print("Saved placeholder ROC (no samples):", out_path)
        return

    if len(np.unique(y_true)) < 2:
        # single-class: save placeholder
        plt.figure(figsize=(4,3))
        plt.text(0.5, 0.5, "ROC not available (single class)", ha='center', va='center')
        plt.axis('off')
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close()
        print("Saved placeholder ROC (single class):", out_path)
        return

    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6,6))
    plt.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.4f}")
    plt.plot([0,1],[0,1], linestyle='--', color='gray')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title or "ROC Curve")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved ROC:", out_path)

# ------------- ensure model loaded -------------
if "fed_model" in globals():
    model = fed_model
    model.to(DEVICE)
    model.eval()
    print("Using existing fed_model in runtime.")
else:
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"Model file not found at {MODEL_PATH}. Load fed_model or place .pth at this path.")
    print("Loading ViT model from:", MODEL_PATH)
    IMG_SIZE = 64  # adjust if needed
    model = ViTForImageClassification.from_pretrained(
        "google/vit-base-patch16-224-in21k",
        num_labels=2,
        image_size=IMG_SIZE,
        ignore_mismatched_sizes=True,
    )
    state = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(state, strict=False)
    model.to(DEVICE)
    model.eval()
    print("Model loaded and ready.")

# ------------- require validation loaders -------------
if "client_val_loaders" not in globals():
    raise RuntimeError("client_val_loaders not found in the runtime. Create data loaders first.")

# ------------- run per-client scoring and save ROC -------------
for idx, val_loader in enumerate(client_val_loaders):
    client_name = f"Client{idx+1}"
    print(f"\n--- Generating ROC for {client_name} ---")
    y_true, y_scores = eval_get_scores(model, val_loader, device=DEVICE)
    out_file = os.path.join(PLOTS_DIR, f"roc_{client_name}.png")
    plot_roc_and_save(y_true, y_scores, out_file, title=f"{client_name} ROC Curve")

print("\nAll ROC plots saved to:", PLOTS_DIR)

Using existing fed_model in runtime.

--- Generating ROC for Client1 ---


Saved ROC: Results\plots_from_model\roc_Client1.png

--- Generating ROC for Client2 ---


Saved ROC: Results\plots_from_model\roc_Client2.png

All ROC plots saved to: Results\plots_from_model


Explainable AI

1 — Load Model + Create ExplainableAI Folder

In [22]:
# Cell 1: load model + utilities + create ExplainableAI folder
import os, math, numpy as np, cv2
from PIL import Image
import torch
import matplotlib.pyplot as plt
from torchvision import transforms
from transformers import ViTForImageClassification

RESULTS_DIR = "Results"
EXPLAIN_DIR = os.path.join(RESULTS_DIR, "ExplainableAI")
os.makedirs(EXPLAIN_DIR, exist_ok=True)

MODEL_PATH = os.path.join(RESULTS_DIR, "vit_monkeypox_federated.pth")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Explainable AI output folder:", EXPLAIN_DIR)
print("Model path exists:", os.path.exists(MODEL_PATH))

# Load ViT model
IMG_SIZE = 64   # adjust to your training size
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=2,
    image_size=IMG_SIZE,
    ignore_mismatched_sizes=True,
)
state = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state, strict=False)
model.to(DEVICE)
model.eval()
print("ViT model loaded and set to eval()")

# Patch size detection
patch_size = getattr(model.config, "patch_size", 16)
if isinstance(patch_size, (tuple, list)):
    patch_size = patch_size[0]
grid_size = IMG_SIZE // patch_size
print("patch_size:", patch_size, "| grid_size:", grid_size)

# Image preprocessing
preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5)),
])

def overlay_heatmap_on_image(orig_pil, heatmap, out_path, alpha=0.5, colormap=cv2.COLORMAP_JET):
    orig = np.array(orig_pil.resize((heatmap.shape[1], heatmap.shape[0])))
    orig_bgr = cv2.cvtColor(orig, cv2.COLOR_RGB2BGR)
    hm_uint8 = np.uint8(255 * heatmap)
    heat_color = cv2.applyColorMap(hm_uint8, colormap)
    overlay = cv2.addWeighted(orig_bgr, 1.0 - alpha, heat_color, alpha, 0)
    cv2.imwrite(out_path, overlay)
    print("Saved:", out_path)


Device: cpu
Explainable AI output folder: Results\ExplainableAI
Model path exists: True


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized because the shapes did not match:
- vit.embeddings.position_embeddings: found shape torch.Size([1, 197, 768]) in the checkpoint and torch.Size([1, 17, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViT model loaded and set to eval()
patch_size: 16 | grid_size: 4


2 — ViT Grad-CAM & Grad-CAM++ Implementation (Updated Folder)

In [23]:
# Cell 2: Grad-CAM and Grad-CAM++ for ViT
class _Hook:
    def __init__(self, module):
        self.activations = None
        self.gradients = None
        module.register_forward_hook(self._forward_hook)
        if hasattr(module, "register_full_backward_hook"):
            module.register_full_backward_hook(self._backward_hook)
        else:
            module.register_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inp, out):
        self.activations = out.detach()

    def _backward_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

class ViTGradCAM:
    def __init__(self, model, target_layer=None):
        if target_layer is None:
            target_layer = model.vit.encoder.layer[-1].output
        self.model = model
        self.hook = _Hook(target_layer)

    def _compute_token_importance(self):
        grads = self.hook.gradients.cpu().numpy()[0]     # (seq, dim)
        acts  = self.hook.activations.cpu().numpy()[0]   # (seq, dim)
        weights = grads.mean(axis=0)
        token_importance = np.maximum((acts * weights).sum(axis=1), 0)
        return token_importance

    def generate(self, input_tensor):
        out = self.model(input_tensor)
        logits = out.logits if hasattr(out, "logits") else out
        pred = torch.argmax(logits, dim=1).item()

        self.model.zero_grad()
        logits[0, pred].backward(retain_graph=False)

        token_importance = self._compute_token_importance()

        token_importance = token_importance[1:]  # remove CLS
        gs = int(math.sqrt(len(token_importance)))
        token_importance = token_importance[: gs * gs]
        cam = token_importance.reshape(gs, gs)

        cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        cam = (cam - cam.min()) / (cam.max() + 1e-8)
        return cam, pred

class ViTGradCAMPlusPlus(ViTGradCAM):
    def _compute_token_importance(self):
        grads = self.hook.gradients.cpu().numpy()[0]
        acts  = self.hook.activations.cpu().numpy()[0]

        grads_p2 = grads ** 2
        grads_p3 = grads ** 3

        denom = 2 * grads_p2 + (acts * grads_p3).sum(axis=1, keepdims=True)
        denom = np.where(denom == 0, 1e-8, denom)

        alpha = grads_p2 / denom
        relu_grads = np.maximum(grads, 0)
        weights = (alpha * relu_grads).sum(axis=0)

        token_importance = np.maximum((acts * weights).sum(axis=1), 0)
        return token_importance


Generate Explainable AI Heatmaps

In [25]:
# Cell 3: Generate explainability maps for ONE image (Grad-CAM + Grad-CAM++)
image_path = "Clients/ClientB/val/monkeypox/M01_01_00.jpg"   # CHANGE THIS

print("Input:", image_path)
pil_img = Image.open(image_path).convert("RGB")
tensor_in = preprocess(pil_img).unsqueeze(0).to(DEVICE)

# --- Grad-CAM ---
cam_gen = ViTGradCAM(model)
cam, pred = cam_gen.generate(tensor_in)

out_gc = os.path.join(EXPLAIN_DIR, f"GradCAM_pred{pred}.png")
overlay_heatmap_on_image(pil_img, cam, out_gc)
print("Predicted class (Grad-CAM):", pred)

# --- Grad-CAM++ ---
campp_gen = ViTGradCAMPlusPlus(model)
cam_pp, pred_pp = campp_gen.generate(tensor_in)

out_gcpp = os.path.join(EXPLAIN_DIR, f"GradCAMPP_pred{pred_pp}.png")
overlay_heatmap_on_image(pil_img, cam_pp, out_gcpp)
print("Predicted class (Grad-CAM++):", pred_pp)

print("\nAll Explainable AI outputs saved to:", EXPLAIN_DIR)


Input: Clients/ClientB/val/monkeypox/M01_01_00.jpg
Saved: Results\ExplainableAI\GradCAM_pred1.png
Predicted class (Grad-CAM): 1
Saved: Results\ExplainableAI\GradCAMPP_pred1.png
Predicted class (Grad-CAM++): 1

All Explainable AI outputs saved to: Results\ExplainableAI


In [26]:
# Cell 3: Generate explainability maps for ONE image (Grad-CAM + Grad-CAM++)
image_path = "Clients/ClientB/val/non_monkeypox/NM01_01_05.jpg"   # CHANGE THIS

print("Input:", image_path)
pil_img = Image.open(image_path).convert("RGB")
tensor_in = preprocess(pil_img).unsqueeze(0).to(DEVICE)

# --- Grad-CAM ---
cam_gen = ViTGradCAM(model)
cam, pred = cam_gen.generate(tensor_in)

out_gc = os.path.join(EXPLAIN_DIR, f"GradCAM_pred{pred}.png")
overlay_heatmap_on_image(pil_img, cam, out_gc)
print("Predicted class (Grad-CAM):", pred)

# --- Grad-CAM++ ---
campp_gen = ViTGradCAMPlusPlus(model)
cam_pp, pred_pp = campp_gen.generate(tensor_in)

out_gcpp = os.path.join(EXPLAIN_DIR, f"GradCAMPP_pred{pred_pp}.png")
overlay_heatmap_on_image(pil_img, cam_pp, out_gcpp)
print("Predicted class (Grad-CAM++):", pred_pp)

print("\nAll Explainable AI outputs saved to:", EXPLAIN_DIR)


Input: Clients/ClientB/val/non_monkeypox/NM01_01_05.jpg
Saved: Results\ExplainableAI\GradCAM_pred0.png
Predicted class (Grad-CAM): 0
Saved: Results\ExplainableAI\GradCAMPP_pred0.png
Predicted class (Grad-CAM++): 0

All Explainable AI outputs saved to: Results\ExplainableAI


WIth Large Images

A — Imports, folders, load model

In [29]:
# Part A — imports, folders, load model
import os, math, numpy as np, cv2
from PIL import Image
import torch
from torchvision import transforms
from transformers import ViTForImageClassification

# --- paths & folders ---
RESULTS_DIR = "Results"
MODEL_PATH = os.path.join(RESULTS_DIR, "vit_monkeypox_federated.pth")
EXPLAIN_DIR = os.path.join(RESULTS_DIR, "ExplainableAI")
os.makedirs(EXPLAIN_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VIS_SIZE = 224   # final visualization size (224 x 224)

print("Device:", DEVICE)
print("Explainable outputs will be saved to:", EXPLAIN_DIR)
print("Model path exists:", os.path.exists(MODEL_PATH))

FALLBACK_IMG_SIZE = 64
# --- load model (use fed_model if present in runtime) ---
if "fed_model" in globals():
    model = fed_model
    model.to(DEVICE)
    model.eval()
    print("Using existing fed_model in memory.")
else:
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"Model not found at {MODEL_PATH}. Place the .pth or create fed_model.")
    # instantiate a ViT skeleton (ignore size mismatches)
    FALLBACK_IMG_SIZE = 64
    model = ViTForImageClassification.from_pretrained(
        "google/vit-base-patch16-224-in21k",
        num_labels=2,
        image_size=FALLBACK_IMG_SIZE,
        ignore_mismatched_sizes=True,
    )
    state = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(state, strict=False)
    model.to(DEVICE)
    model.eval()
    print("Loaded ViT model from disk.")

# Determine model input size if present in config; else fall back
try:
    MODEL_IMG_SIZE = int(getattr(model.config, "image_size", getattr(model.config, "input_size", FALLBACK_IMG_SIZE)))
except Exception:
    MODEL_IMG_SIZE = FALLBACK_IMG_SIZE
print("MODEL_IMG_SIZE used for prediction (keep same as training):", MODEL_IMG_SIZE)


Device: cpu
Explainable outputs will be saved to: Results\ExplainableAI
Model path exists: True
Using existing fed_model in memory.
MODEL_IMG_SIZE used for prediction (keep same as training): 64


B — Hook + ViT Grad-CAM & Grad-CAM++ (lightweight)

In [31]:
# Part B — Hook and ViT Grad-CAM / Grad-CAM++ classes
class _Hook:
    def __init__(self, module):
        self.activations = None
        self.gradients = None
        module.register_forward_hook(self._forward_hook)
        # use full backward hook if available, else fallback
        if hasattr(module, "register_full_backward_hook"):
            module.register_full_backward_hook(self._backward_hook)
        else:
            module.register_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inp, out):
        # out shape expected: (batch, seq_len, hidden_dim)
        self.activations = out.detach()

    def _backward_hook(self, module, grad_in, grad_out):
        # grad_out[0] shape: (batch, seq_len, hidden_dim)
        self.gradients = grad_out[0].detach()

class ViTGradCAM:
    """
    Simple ViT Grad-CAM / Grad-CAM++ generator.
    Attach to the last encoder layer output by default.
    """
    def __init__(self, model, target_module=None):
        self.model = model
        if target_module is None:
            target_module = model.vit.encoder.layer[-1].output
        self.hook = _Hook(target_module)

    def _token_importance_gradcam(self):
        grads = self.hook.gradients.cpu().numpy()[0]   # (seq, dim)
        acts  = self.hook.activations.cpu().numpy()[0] # (seq, dim)
        weights = np.mean(grads, axis=0)               # (dim,)
        token_imp = np.maximum((acts * weights).sum(axis=1), 0)  # (seq,)
        return token_imp

    def _token_importance_gradcampp(self):
        grads = self.hook.gradients.cpu().numpy()[0]
        acts  = self.hook.activations.cpu().numpy()[0]
        g2 = grads ** 2
        g3 = grads ** 3
        denom = 2 * g2 + (acts * g3).sum(axis=1, keepdims=True)
        denom = np.where(denom == 0, 1e-8, denom)
        alpha = g2 / denom
        relu_grads = np.maximum(grads, 0.0)
        weights = (alpha * relu_grads).sum(axis=0)
        token_imp = np.maximum((acts * weights).sum(axis=1), 0)
        return token_imp

    def generate(self, input_tensor, method="gradcam", target_class=None):
        """
        input_tensor: torch tensor shape (1,C,H,W) with H,W == MODEL_IMG_SIZE used for prediction
        method: "gradcam" or "gradcampp"
        returns: cam_resized (MODEL_IMG_SIZE x MODEL_IMG_SIZE) normalized [0,1], predicted_class (int)
        """
        input_tensor = input_tensor.to(next(self.model.parameters()).device)
        outputs = self.model(input_tensor)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs
        pred = torch.argmax(logits, dim=1).item()
        cls = pred if target_class is None else int(target_class)

        self.model.zero_grad()
        logits[0, cls].backward(retain_graph=False)

        if method == "gradcam":
            token_imp = self._token_importance_gradcam()
        else:
            token_imp = self._token_importance_gradcampp()

        # remove CLS token
        token_imp = token_imp[1:]
        # ensure perfect square length
        L = len(token_imp)
        gs = int(math.sqrt(L))
        if gs * gs != L:
            token_imp = token_imp[: gs * gs]
            L = len(token_imp)
            gs = int(math.sqrt(L))
        cam = token_imp.reshape(gs, gs)
        # resize token-grid to MODEL_IMG_SIZE (visual base)
        cam_resized = cv2.resize(cam, (MODEL_IMG_SIZE, MODEL_IMG_SIZE), interpolation=cv2.INTER_CUBIC)
        cam_resized = cam_resized - cam_resized.min()
        if cam_resized.max() > 0:
            cam_resized = cam_resized / cam_resized.max()
        else:
            cam_resized = np.zeros_like(cam_resized)
        return cam_resized, pred


C — Preprocess for model & high-quality 224×224 overlay saver

In [32]:
# Part C — Preprocessing for model input and high-quality overlay saver (224x224)
from torchvision import transforms

# Preprocess used for prediction (keeps predictions consistent)
preprocess_for_model = transforms.Compose([
    transforms.Resize((MODEL_IMG_SIZE, MODEL_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5)),
])

def save_overlay_vis224(orig_path, heatmap, out_path, alpha=0.5, colormap=cv2.COLORMAP_JET, vis_size=VIS_SIZE):
    """
    heatmap: 2D numpy array in [0,1], sized MODEL_IMG_SIZE x MODEL_IMG_SIZE (or any)
    Saves an overlay PNG at vis_size x vis_size (224x224 by default).
    """
    pil = Image.open(orig_path).convert("RGB")
    # Resize original to 224x224 for visualization (bicubic)
    orig_resized = pil.resize((vis_size, vis_size), resample=Image.BICUBIC)
    orig_np = np.array(orig_resized)
    orig_bgr = cv2.cvtColor(orig_np, cv2.COLOR_RGB2BGR)

    # Resize heatmap to vis_size with bicubic
    hm_uint8 = np.uint8(255 * heatmap)
    hm_resized = cv2.resize(hm_uint8, (vis_size, vis_size), interpolation=cv2.INTER_CUBIC)
    heat_color = cv2.applyColorMap(hm_resized, colormap)

    overlay = cv2.addWeighted(orig_bgr, 1.0 - alpha, heat_color, alpha, 0)
    # Save with low compression for high visual quality
    cv2.imwrite(out_path, overlay, [int(cv2.IMWRITE_PNG_COMPRESSION), 1])
    print("Saved high-quality overlay:", out_path)


D — Generate Grad-CAM & Grad-CAM++ for ONE image

In [33]:
# Part D — generate Grad-CAM and Grad-CAM++ for a single image (224x224 outputs)
image_path = "Clients/ClientB/val/monkeypox/M01_01_00.jpg"   # <-- change to the image you want
if not os.path.exists(image_path):
    raise FileNotFoundError("image not found: " + image_path)

# prepare model input (MODEL_IMG_SIZE)
pil_img = Image.open(image_path).convert("RGB")
input_tensor = preprocess_for_model(pil_img).unsqueeze(0).to(DEVICE)

# instantiate CAM generator (hooks last encoder by default)
cam_gen = ViTGradCAM(model)

# Grad-CAM (vanilla)
cam_v, pred_v = cam_gen.generate(input_tensor, method="gradcam")
out_v = os.path.join(EXPLAIN_DIR, f"GradCAM_224_pred{pred_v}.png")
save_overlay_vis224(image_path, cam_v, out_v, alpha=0.5)

# Grad-CAM++ (sharper)
cam_pp, pred_pp = cam_gen.generate(input_tensor, method="gradcampp")
out_pp = os.path.join(EXPLAIN_DIR, f"GradCAMPP_224_pred{pred_pp}.png")
save_overlay_vis224(image_path, cam_pp, out_pp, alpha=0.5)

print("Predictions -> GradCAM:", pred_v, ", GradCAM++:", pred_pp)
print("Saved to:", EXPLAIN_DIR)


Saved high-quality overlay: Results\ExplainableAI\GradCAM_224_pred1.png
Saved high-quality overlay: Results\ExplainableAI\GradCAMPP_224_pred1.png
Predictions -> GradCAM: 1 , GradCAM++: 1
Saved to: Results\ExplainableAI


In [34]:
# Part D — generate Grad-CAM and Grad-CAM++ for a single image (224x224 outputs)
image_path = "Clients/ClientB/val/non_monkeypox/NM01_01_05.jpg"   # <-- change to the image you want
if not os.path.exists(image_path):
    raise FileNotFoundError("image not found: " + image_path)

# prepare model input (MODEL_IMG_SIZE)
pil_img = Image.open(image_path).convert("RGB")
input_tensor = preprocess_for_model(pil_img).unsqueeze(0).to(DEVICE)

# instantiate CAM generator (hooks last encoder by default)
cam_gen = ViTGradCAM(model)

# Grad-CAM (vanilla)
cam_v, pred_v = cam_gen.generate(input_tensor, method="gradcam")
out_v = os.path.join(EXPLAIN_DIR, f"GradCAM_224_pred{pred_v}.png")
save_overlay_vis224(image_path, cam_v, out_v, alpha=0.5)

# Grad-CAM++ (sharper)
cam_pp, pred_pp = cam_gen.generate(input_tensor, method="gradcampp")
out_pp = os.path.join(EXPLAIN_DIR, f"GradCAMPP_224_pred{pred_pp}.png")
save_overlay_vis224(image_path, cam_pp, out_pp, alpha=0.5)

print("Predictions -> GradCAM:", pred_v, ", GradCAM++:", pred_pp)
print("Saved to:", EXPLAIN_DIR)


Saved high-quality overlay: Results\ExplainableAI\GradCAM_224_pred0.png
Saved high-quality overlay: Results\ExplainableAI\GradCAMPP_224_pred0.png
Predictions -> GradCAM: 0 , GradCAM++: 0
Saved to: Results\ExplainableAI


In [35]:
from PIL import Image
import cv2
import numpy as np
import os

# ---------- INPUT PATHS ----------
original_path = r"Clients/ClientB/val/monkeypox/M01_01_00.jpg"
heatmap_path  = r"Results/ExplainableAI/GradCAM_224_pred1.png"

# ---------- OUTPUT PATHS ----------
output_side_by_side = r"Results/ExplainableAI/merged_side_by_side.png"
output_overlay      = r"Results/ExplainableAI/merged_overlay.png"

# ---------- LOAD IMAGES ----------
original = Image.open(original_path).convert("RGB")
heatmap  = Image.open(heatmap_path).convert("RGB")

# Resize original to match heatmap (usually 224x224)
original_resized = original.resize(heatmap.size, Image.BICUBIC)

# ============================================================
# 1️⃣ SIDE-BY-SIDE MERGE : (Original | Heatmap)
# ============================================================
w, h = heatmap.size
side = Image.new("RGB", (w * 2, h))

side.paste(original_resized, (0, 0))
side.paste(heatmap, (w, 0))

os.makedirs(os.path.dirname(output_side_by_side), exist_ok=True)
side.save(output_side_by_side)
print("Saved Side-by-Side Image:", output_side_by_side)

# ============================================================
# 2️⃣ OVERLAY MERGE : (Original + Heatmap Blend)
# ============================================================
orig_np = np.array(original_resized)
heat_np = np.array(heatmap)

# Convert RGB → BGR for OpenCV blending
orig_bgr = cv2.cvtColor(orig_np, cv2.COLOR_RGB2BGR)
heat_bgr = cv2.cvtColor(heat_np, cv2.COLOR_RGB2BGR)

# Blend 50/50
overlay_bgr = cv2.addWeighted(orig_bgr, 0.5, heat_bgr, 0.5, 0)
overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)

overlay_img = Image.fromarray(overlay_rgb)
overlay_img.save(output_overlay)

print("Saved Overlay Image:", output_overlay)


Saved Side-by-Side Image: Results/ExplainableAI/merged_side_by_side.png
Saved Overlay Image: Results/ExplainableAI/merged_overlay.png
